# 07 · Memory: Conversations 🧠

Every `Runner.run` starts with a blank mind. To remember a chat, the Agents SDK can save it in an **OpenAI Conversation** — a chat history stored on OpenAI's servers.
We only keep its ID (like `conv_abc123`).

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv

if not os.path.exists("main.py"):   # make sure we're in the project folder
    os.chdir("../..")

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

MODEL = os.environ.get("OPENAI_MODEL") or "gpt-6-luna"
print("Ready! Using", MODEL)

In [ ]:
from agents import Agent, Runner

uncle = Agent(
    name="Kopi Uncle",
    instructions="You are a friendly kopitiam uncle. Keep replies short.",
    model=MODEL,
)

## 1. Without memory 🐟

In [ ]:
await Runner.run(uncle, "Hi Uncle, my name is Ahmad and I always drink teh-c.")

result = await Runner.run(uncle, "What's my name and my usual drink?")
print(result.final_output)

Uncle forgot already! Each run is a new chat.

## 2. With memory: a session 💾

In [ ]:
from agents import OpenAIConversationsSession

session = OpenAIConversationsSession()   # a brand-new conversation

await Runner.run(uncle, "Hi Uncle, my name is Ahmad and I always drink teh-c.", session=session)
result = await Runner.run(uncle, "What's my name and my usual drink?", session=session)

print(result.final_output)

In [ ]:
# this ID is all we need to remember
conversation_id = session.session_id
print(conversation_id)

## 3. Peek inside the conversation

In [ ]:
items = await session.get_items()

for item in items:
    if item.get("type") == "message":
        text = " ".join(c.get("text", "") for c in item["content"])
        print(f"{item['role']:>9}: {text}")

## 4. Come back later
Pretend the program restarted. With just the ID, we can continue the chat.

In [ ]:
later = OpenAIConversationsSession(conversation_id=conversation_id)

result = await Runner.run(uncle, "Same as usual, Uncle!", session=later)
print(result.final_output)

## 5. Many customers at once 👥
Each Telegram chat needs its **own** conversation. A dictionary maps `chat_id -> conversation_id`.

In [ ]:
conversations = {}   # chat_id -> conversation_id


async def chat(chat_id, text):
    session = OpenAIConversationsSession(conversation_id=conversations.get(chat_id))
    result = await Runner.run(uncle, text, session=session)
    conversations[chat_id] = session.session_id   # remember it for next time
    return result.final_output


print(await chat(111, "I'm Mei Ling, I like kopi siew dai."))
print(await chat(222, "I'm Priya, I like milo."))
print(await chat(111, "What do I like?"))
print(await chat(222, "What do I like?"))

In [ ]:
conversations

## 🤔 The problem
`conversations` lives in your computer's memory. Restart the notebook (or the bot) and it's **gone** — every customer becomes a stranger again.

**Next notebook:** we save it in a real database. 🗄️

## ✏️ Exercises
1. Add a `/reset` feature: if the text is `/reset`, delete that chat from `conversations`.
2. Use `await session.get_items()` to count how many messages a chat has.
3. Ask Uncle *"what did I order just now?"* after ordering something — does he remember?

In [ ]:
# your code here